# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-azeemi/fatima-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



> Finding 1: The "Freshness Multiplier"Claim in Paper: Refreshing content older than 365 days yields a $3.2\times$ higher health score and up to $57\times$ higher impressions compared to stale content.Methodology Question: Where does the label come from and how was selection bias handled?Constructive Review: Since pages targeted for a refresh are typically hand-picked by human editors (who naturally choose pages with existing potential or high historical relevance), part of the performance jump ($3.2\times$) is likely driven by the selection bias of which pages got refreshed rather than the edit intervention alone. To make this claim robust, we should separate "randomly chosen refreshes" from "strategic high-priority refreshes" or explicitly state that this is an association observed in retrospective portfolio data.


> Finding 2: The Decay CliffClaim in Paper: Content hits a peak performance window at 61–90 days, followed by a sharp "decay cliff" where the health score drops significantly by days 271–365.Methodology Question: Does the validation design account for cohort or temporal bias across different publishing seasons?Constructive Review: If older content (e.g., published a year ago) belongs to a different seasonal or content cohort than newer content (published 2 months ago), the "decay" might reflect macroscopic shifts in search intent or seasonality rather than pure content aging. Tracking the exact same URLs longitudinally over a 365-day lifecycle using a time-aware split would ensure we are measuring true content decay instead of cohort baseline differences.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import pandas as pd
import numpy as np
import os, sys
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Setup repository path safely for Google Colab if needed
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# 1. Load dataset locally
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['target'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'word_count',
    'content_age_days',
    'impressions_90d',
    'avg_position',
    'ctr',
    'search_volume'
]
available_features = [col for col in feature_cols if col in df.columns]

X = df[available_features]
y = df['target']

# 2. Split FIRST (Honest Split: Train/Test Isolation)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Impute missing values SEPARATELY using Training Median (No Leakage)
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=available_features)
X_test_imp = pd.DataFrame(imputer.transform(X_test_raw), columns=available_features)

# 4. Train Model
honest_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
honest_model.fit(X_train_imp, y_train)

# 5. Predict on both Train and Test to check performance & overfitting
train_preds = honest_model.predict(X_train_imp)
train_probs = honest_model.predict_proba(X_train_imp)[:, 1]

test_preds = honest_model.predict(X_test_imp)
test_probs = honest_model.predict_proba(X_test_imp)[:, 1]

# 6. Metrics comparison function
def get_full_metrics(y_true, y_pred, y_prob):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0),
        roc_auc_score(y_true, y_prob)
    ]

train_mets = get_full_metrics(y_train, train_preds, train_probs)
test_mets = get_full_metrics(y_test, test_preds, test_probs)

# 7. Print Final Comparison Table
split_comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Training Set': train_mets,
    'Test Set (Unseen)': test_mets
})

print("=== HONEST SPLIT: TRAINING VS TESTING PERFORMANCE ===")
print(split_comparison_df.to_string(index=False))

=== HONEST SPLIT: TRAINING VS TESTING PERFORMANCE ===
   Metric  Training Set  Test Set (Unseen)
 Accuracy      0.698958           0.676833
Precision      0.676598           0.662621
   Recall      0.851806           0.822571
 F1-Score      0.754160           0.733983
  ROC-AUC      0.773070           0.735929


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

> Feature Leakage Check: Verified that all active features (word_count, content_age_days, impressions_90d, avg_position, ctr, search_volume) are calculated strictly from pre-window operational baselines and do not contain post-target outcome timelines.


> Imputation Isolation: Missing values were handled using SimpleImputer (median strategy), which was fitted strictly on the training split (X_train) and then transformed separately on the test split (X_test). This completely prevents any test set statistics or information from leaking into the training pipeline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



> Rewritten Honest Claim: "Using an honest train-test split with isolated missing-value imputation, the Random Forest model achieved an observed ROC-AUC of 0.7359 on unseen test data. While historical traffic and content age exhibit a strong directional association with content performance declines, these metrics serve as decision-support insights for prioritizing editorial reviews rather than definitive causal guarantees."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Charts for the report

*Save the train-vs-test gap chart used in `outputs/model_report.md`.*

In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return cand
    return here

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import simple_svg_bar_chart

# split_comparison_df from the honest-split cell
row = {r["Metric"]: r for r in split_comparison_df.to_dict(orient="records")}
labels = ["Train Acc", "Test Acc", "Train F1", "Test F1", "Train ROC", "Test ROC"]
values = [
    float(row["Accuracy"]["Training Set"]) * 100,
    float(row["Accuracy"]["Test Set (Unseen)"]) * 100,
    float(row["F1-Score"]["Training Set"]) * 100,
    float(row["F1-Score"]["Test Set (Unseen)"]) * 100,
    float(row["ROC-AUC"]["Training Set"]) * 100,
    float(row["ROC-AUC"]["Test Set (Unseen)"]) * 100,
]

for folder in (ROOT / "work" / "outputs" / "charts", ROOT / "outputs" / "charts"):
    folder.mkdir(parents=True, exist_ok=True)
    simple_svg_bar_chart(
        "RF train vs test (honest gap check)",
        labels,
        values,
        folder / "train_vs_test.svg",
        color="#B85C38",
    )

print("Saved train_vs_test.svg")
print(split_comparison_df.to_string(index=False))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.